In [46]:
import pandas as pd

df = pd.read_csv("Admission_Predict.csv")
print(df.shape)
df.head()

(400, 9)


,Serial No.,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
0,1,337,118,4,4.5,4.5,9.65,1,0.92
1,2,324,107,4,4.0,4.5,8.87,1,0.76
2,3,316,104,3,3.0,3.5,8.00,1,0.72
3,4,322,110,3,3.5,2.5,8.67,1,0.80
4,5,314,103,2,2.0,3.0,8.21,0,0.65


In [47]:
df = df.drop(columns=["Serial No."], errors="ignore")

In [48]:
X = df.drop("Chance of Admit", axis=1)
y = df["Chance of Admit"]

In [49]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SelectFromModel
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

rf_fs = RandomForestRegressor(
    n_estimators=100,
    max_depth=3,
    min_samples_leaf=15,
    min_samples_split=15,
    random_state=42
)
rf_fs.fit(X_train_scaled, y_train)

# Select important features only
selector = SelectFromModel(rf_fs, threshold="median", prefit=True)
X_train_sel = selector.transform(X_train_scaled)
X_test_sel = selector.transform(X_test_scaled)

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=2,
    min_samples_leaf=20,
    min_samples_split=20,
    random_state=42
)
rf.fit(X_train_sel, y_train)

train_pred = rf.predict(X_train_sel)
test_pred = rf.predict(X_test_sel)


In [50]:
train_mse = mean_squared_error(y_train, train_pred)
test_mse = mean_squared_error(y_test, test_pred)
print(f"Train MSE: {train_mse:.3f}")
print(f"Test MSE : {test_mse:.3f}")


Train MSE: 0.004
Test MSE : 0.006


In [51]:
# Model evaluation: R2, MAE, RMSE
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

r2 = r2_score(y_test, test_pred)
mae = mean_absolute_error(y_test, test_pred)
rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print(f"R2 Score : {r2:.3f}")
print(f"MAE      : {mae:.3f}")
print(f"RMSE     : {rmse:.3f}")


R2 Score : 0.776
MAE      : 0.054
RMSE     : 0.076
